### ЗАДАЧА: Пакетная загрузка отгрузок (try/except + custom exceptions)

Из внешней логистической системы приходят строки с отгрузками.
Нужно безопасно распарсить данные, отделить валидные записи от ошибок
и посчитать несколько итоговых метрик.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `ShipmentError`
   - `RowFormatError`
   - `WeightError`
   - `PriorityError`
   - `RegionError`.

2. Функцию `parse_shipment(row)`:
   - формат строки: `shipment_id,client,weight,priority,region`
   - `weight` должен быть числом и `> 0`
   - допустимые приоритеты: `standard`, `express`, `vip`
   - допустимые регионы: `RU`, `KZ`, `BY`
   - при ошибке конвертации веса использовать `raise ... from ...`.

3. Функцию `load_shipments(rows)`:
   - вернуть `(shipments, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных отгрузок,
   - ошибки по типам,
   - суммарный вес только для `express` и `vip`,
   - клиента-лидера по суммарному весу среди валидных записей.

In [ ]:
rows = [
    'S-100,Acme,12.5,express,RU',
    'S-101,Beta,0,standard,RU',
    'S-102,Acme,abc,vip,KZ',
    'S-103,Delta,8.5,urgent,BY',
    'S-104,Gamma,15,vip,UZ',
    'S-105,Acme,4.0,standard,KZ',
    'S-106,Beta,9.5,express,BY',
]

class ShipmentError(Exception):
    pass

class RowFormatError(ShipmentError):
    pass

class WeightError(ShipmentError):
    pass

class PriorityError(ShipmentError):
    pass

class RegionError(ShipmentError):
    pass

def parse_shipment(row):
    # TODO: распарсить строку и провалидировать weight, priority, region
    try:
        parts = row.split(",")
        if len(parts) != 5:
            raise RowFormatError(f"Неверный формат строки: ожидалось 5 полей, получено {len(parts)}")
        
        shipment_id,client,weight_str,priority,region = parts

        try:
            weight = float(weight_str)
            if weight < 0:
                raise WeightError(f"Вес не должен быть отрицательным: {weight}")
            
        # TODO: при ошибке конвертации weight использовать raise ... from ...
        except ValueError as e:
            raise WeightError(f"Недопустимое значение веса: '{weight_str}'") from e
        
        valid_priorities = {"express", "standart", "urgent", "vip"}
        if priority not in valid_priorities:
            raise PriorityError(f"Неверный приоритет: '{priority}'. Допустимое значение: {valid_priorities}")
        
        valid_regions = {"RU", "KZ", "BY", "UZ"}
        if region not in valid_regions:
            raise RegionError(f"Неверный регион: '{region}'. Допустимое значение: {valid_regions}")
        
        return {
            "shipment_id": shipment_id,
            "client": client,
            "weight": weight,
            "priority": priority,
            "region": region
        }
    
    except ShipmentError:
        raise
    except Exception as e:
        raise RowFormatError(f"")

def load_shipments(rows):
    # TODO: вернуть (shipments, errors)
    shipments = []
    errors = []

    for row in rows:
        try:
            shipments.append(parse_shipment(row))
        except ShipmentError as e:
            errors.append((row, type(e).__name__, e))
    return shipments, errors

# TODO: вызвать load_shipments(rows)
shipments, errors = load_shipments(rows)

# TODO: вывести число валидных отгрузок и число ошибок
print(f"Число валидных отгрузок: {len(shipments)}")
print(f"Число ошибок: {len(errors)}")

# TODO: вывести ошибки по типам
error_types = {}
for _,error_type,_ in errors:
    if error_type in error_types:
        error_types[error_type] += 1
    else:
        error_types[error_type] = 1
print(error_types)

# TODO: посчитать premium_weight только для express/vip
premium_priorities = {"express", "vip"}
premium_weight = 0
for shipment in shipments:
    if shipment["priirity"] in premium_priorities:
        premium_weight += shipment["weight"]
print(f"Premium_weight (express/vip): {premium_weight}")

# TODO: найти клиента-лидера по суммарному весу
client_weights = {}
for shipment in shipments:
    client = shipment["client"]
    weight = shipment["weight"]
    if client in client_weights:
        client_weights[client] += weight
    else:
        client_weights[client] = weight

if client_weights:
    leader_client = None
    leader_weight = -1
    for clietn, total_weight in client_weights.items():
        if total_weight > leader_weight:
            leader_weight = total_weight
            leader_client = client
    print(f"Клиент-лидер: {leader_client} с суммарным весом {leader_weight}")
else:
    print("Нет валидных отгрузок для анализа клиентов")